# 01 · Tuning (validation only)

Every setting the paper leaves open is chosen here, **on the validation split only** -
test numbers are never looked at. The result, `tuned.json`, is read by notebooks 02-07
(on Kaggle: add this notebook's output as an input there).

* **Stage A - models.** For every dataset: our transformer and official SurvTRACE on
  `N_CONFIGS` configurations drawn from SurvTRACE's grid (lr, weight decay, layers,
  embedding size, intermediate size, heads); Cox (ridge penalty), Random Survival Forest,
  DeepSurv and PC-Hazard on their own grids. DeepHit / DSM / MENSA use the transformer
  backbone tuned for our model (same encoder, different head).
* **Stage B - loss weights** (with the tuned backbone): weight and sigma of L_rank (Eq. 4)
  and L_mul (Eq. 5), the L_MM weight (Eq. 7) and the regression
  head's time unit (end of follow-up vs raw time).

Selection: mean validation C_td over `TUNE_SEEDS` (regression-only runs: validation
MAE-margin).

## Options

In [ ]:
NAME = "01_tuning"

SEEDS = list(range(10))   # protocol: 10 random 60/10/30 splits (as SurvTRACE); seed s = split s and init s

TUNE_SEEDS = [0]          # splits used for tuning
DATASETS = ["metabric", "support", "ebmt", "hsa_synthetic", "deephit_synthetic"]
N_CONFIGS = 24            # configurations drawn from the 216-point SurvTRACE grid
N_CONFIGS_NN = 16         # DeepSurv / PC-Hazard (36-point grid)
MODELS = ["ours", "survtrace", "cox", "rsf", "deepsurv", "pchazard"]
RANK_GRID = {"coeff": [0.1, 0.5, 1.0], "sigma": [0.1, 0.5, 1.0]}
MM_COEFF = [0.1, 0.5, 1.0]

# --- execution ------------------------------------------------------------------------
SMOKE_TEST   = False    # True: 2 epochs, 1 seed, 1-2 configurations - only to check that it runs
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h);
                            # re-run with this notebook's output attached to continue
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # the code is cloned from here
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    RESULTS_ROOT = WORK / "results"
    INPUT_ROOTS = [Path("/kaggle/input")]
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "version_2"
    INPUT_ROOTS = []
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
print("repo:", REPO, "| commit:", COMMIT, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy and torch are NEVER moved: pandas/pyarrow and CUDA in the image are built for them.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            "pycox==0.3.0", "peft", "easydict", "lifelines", "osqp", "patsy",
            f"numpy=={_np.__version__}", f"torch=={torch.__version__.split('+')[0]}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    # SurvivalEVAL declares torch>=2.13; it runs on older torch, so its pins are skipped
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "SurvivalEVAL==0.8.5"], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
import pandas as pd
from scripts.runner import Runner, select_best
from scripts import v2

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input") and finished runs are
# copied back, so only the missing ones are trained.
n_copied = 0
for root in INPUT_ROOTS:
    for d in root.rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "commit": COMMIT, "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Data (built once, label files per split seed)

In [ ]:
if SMOKE_TEST:
    TUNE_SEEDS, N_CONFIGS, N_CONFIGS_NN = TUNE_SEEDS[:1], 2, 2
    DATASETS = DATASETS[:1] + ["ebmt"]
v2.prepare(r, DATASETS, sorted(set(TUNE_SEEDS)), REPO)

## Stage A · models

In [ ]:
runs = []
for name in DATASETS:
    for model in MODELS:
        n = N_CONFIGS if model in ("ours", "survtrace") else (N_CONFIGS_NN if model in ("deepsurv", "pchazard") else None)
        configs = v2.sample_grid(v2.TUNE_GRIDS[model], n, seed=0)
        runs += v2.tune_runs(model, name, configs, TUNE_SEEDS, REPO)
print(len(runs), "tuning runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

TUNED = {name: {} for name in DATASETS}
for name in DATASETS:
    for model in MODELS:
        try:
            best = select_best(r, f"{name}__{model}__tune")
            TUNED[name][model] = best["config"]
            print(f"{name:18s} {model:10s} val C_td {best['score']:.4f}  {best['config']}")
        except RuntimeError as e:
            print(f"{name:18s} {model:10s} NOT TUNED ({e}) - defaults will be used")

## Stage B · loss weights (tuned backbone)

In [ ]:
def ours(name):
    return v2.tuned_overrides(TUNED, "ours", name)

runs = []
rank_cfgs = v2.grid(RANK_GRID) if not SMOKE_TEST else v2.grid(RANK_GRID)[:2]
for name in DATASETS:
    multi = v2.DATASETS[name]["events"] > 1
    runs += v2.tune_runs("ours", name, [{"v2_rank_coeff": c["coeff"], "v2_rank_sigma": c["sigma"]} for c in rank_cfgs],
                         TUNE_SEEDS, REPO, extra=ours(name), prefix=f"{name}__rank__tune")
    # regression head time unit: end of follow-up (default) vs raw time, R2 regression-only
    reg = ours(name) + ["tasks=v2_regression", "selection_metric=eval_reg_mae_margin", "selection_greater=false",
                        "v2_l1_type=margin"]
    runs += v2.tune_runs("ours", name, [{}, {"v2_time_scale": 1.0}], TUNE_SEEDS, REPO, extra=reg,
                         prefix=f"{name}__timescale__tune")
    if multi:
        runs += v2.tune_runs("ours", name, [{"v2_mul_coeff": c["coeff"], "v2_mul_sigma": c["sigma"]} for c in rank_cfgs],
                             TUNE_SEEDS, REPO, extra=ours(name), prefix=f"{name}__mul__tune")
        runs += v2.tune_runs("ours", name, [{"v2_mm_coeff": c, "v2_mm_t_q": "observed"} for c in MM_COEFF],
                             TUNE_SEEDS, REPO, extra=reg, prefix=f"{name}__mm__tune")
print(len(runs), "loss-tuning runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

def pick(prefix, metric="ctd_weighted_avg", greater=True, default=None):
    try:
        b = select_best(r, prefix, metric, greater)
        print(f"{prefix:32s} {b['config']}  (val {metric} {b['score']:.4f})")
        return b["config"]
    except RuntimeError:
        print(f"{prefix:32s} not available - default {default}")
        return default

TUNED["losses"] = {}
for name in DATASETS:
    multi = v2.DATASETS[name]["events"] > 1
    rk = pick(f"{name}__rank__tune", default={"v2_rank_coeff": 0.5, "v2_rank_sigma": 0.5})
    ts = pick(f"{name}__timescale__tune", "reg_mae_margin", False, default={})
    L = {"rank": {"coeff": rk["v2_rank_coeff"], "sigma": rk["v2_rank_sigma"]},
         "time_scale_raw": bool(ts.get("v2_time_scale") == 1.0)}
    if multi:
        mu = pick(f"{name}__mul__tune", default={"v2_mul_coeff": 0.5, "v2_mul_sigma": 0.5})
        mm = pick(f"{name}__mm__tune", "reg_mae_margin", False, default={"v2_mm_coeff": 0.5})
        L["mul"] = {"coeff": mu["v2_mul_coeff"], "sigma": mu["v2_mul_sigma"]}
        L["mm_coeff"] = mm["v2_mm_coeff"]
    TUNED["losses"][name] = L

(r.results_dir / "tuned.json").write_text(json.dumps(TUNED, indent=2))
print(json.dumps(TUNED, indent=1))

## Tables + zip

In [ ]:
rep = r.results_dir / "report"
rep.mkdir(exist_ok=True)
val = r.collect(split="validation")
val.to_csv(rep / "tuning_validation.csv", index=False)
print("zip:", r.zip())

## Clean up

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)